<a href="https://github.com/BikeNetKit"><img src="media/logo_bikenetkit.png" width="260px" alt="BikeNetKit logo"/></a>

Source: [https://github.com/BikeNetKit/tutorials/tree/main/python/tutorial01](https://github.com/BikeNetKit/tutorials/tree/main/python/tutorial01)

# Tutorial 1 
**Introduction to GrowBikeNet** · *15 Minutes*  

Presented the first time at [D3A](https://d3aconference.dk/data-science-of-human-centric-mobility/), Nyborg, Oct 8th 2026.  

Using GrowBikeNet v0.14.5  
Created 2026-09-30 by Michael Szell: contact@bikenetkit.org

## Prerequisities

- You have [conda](https://docs.conda.io/projects/conda/en/latest/index.html) installed (or the faster [mamba](https://mamba.readthedocs.io/en/latest/index.html)).
- You have a UNIX/Mac-based terminal.

## Setup

First, download and unzip the `tutorial01` folder: [**Download zip file**](https://downgit.github.io/#/home?url=https://github.com/BikeNetKit/tutorials/tree/main/python/tutorial01).  
(Or manually download all contents: [https://github.com/BikeNetKit/tutorials/tree/main/python/tutorial01](https://github.com/BikeNetKit/tutorials/tree/main/python/tutorial01))

Then use the terminal to navigate to this folder. 

Install python, growbikenet, and fixbikenet via:
```
conda env create --file environment.yml
``` 
Then, activate the environment via:
```
conda activate bikenetkit_tutorials 
```
and run jupyter lab:
```
jupyter lab tutorial01.ipynb
```
Now, a browser window with this tutorial should open.

## Introduction to GrowBikeNet

The standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

We run it on an example city, **Nyborg - an important meeting hub for data scientists**. By default, GrowBikeNet just needs one parameter, the city name, which it uses to fetch the city's data from an overpass server online:  
`edges_ordered = gbn.growbikenet("Nyborg Municipality")`

However, this download can take 2 minutes. Also, this server is unfortunately unreliable - especially when multiple people want to download from the same IP address. Also, this selects the whole Nybork municipality, not just the urban area. To prevent the wait and the uncertainty, we make use of GrowBikeNet's data import features, first setting the `import_path`, and importing prepared files via the `import_files` parameter:

### Growing from scratch

We first grow a bike network from scratch:

In [ ]:
gbn.settings.import_path = 'data/'
edges_ordered = gbn.growbikenet(
    "Nyborg",
    import_files={
        'city_boundary': 'nyborg_city_boundary.geojson',
        'growable_network': 'nyborg_growable_network.gpkg',
    }
)

Let's explore the output visually:

In [ ]:
edges_ordered.explore( 
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png",
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 3, "color": "#096a51"},
)

Under the hood, GrowBikeNet sets up a set of seed points, then triangulates them, and routes the links back to the streets:

<img src="media/seedpoints.jpg" width="740px" alt="Seed points"/>

### Growing from existing bicycle network

Growing from scratch is nice, and useful for research. But planners want to know how to grow from their existing network. So let's do that!

Nyborg's network of protected bike lanes, which is provided in the file `nyborg_bike_network.gpkg`, shows some development, but is still quite fragmented:

In [ ]:
import geopandas as gpd
bike_network = gpd.read_file(gbn.settings.import_path+'nyborg_bike_network.gpkg', layer='edges')
bike_network.explore( 
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png",
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 3, "color": "#9999cc"},
)

To fix it, we also need to pass the city's existing bike network, and we pass the parameter `existing_network_spacing='auto'` which tells GrowBikeNet to snap seed points to the existing bike network and to do so automatically:

In [ ]:
edges_ordered = gbn.growbikenet(
    "Nyborg",
    existing_network_spacing='auto',
    import_files={
        'city_boundary': 'nyborg_city_boundary.geojson',
        'growable_network': 'nyborg_growable_network.gpkg',
        'bike_network': 'nyborg_bike_network.gpkg',
    }
)

Let's explore the output:

In [ ]:
edges_ordered

The existing bike network is placed on the first row. We color it purple, while the proposed growing network we color in green:

In [ ]:
import folium

viz = edges_ordered.iloc[:1].explore(
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png", 
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 2, "color": "#9999cc"},
    name="Existing bike network",
)
viz = edges_ordered.iloc[1:].explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#096a51"},
    name="Grown bike network",
)
folium.LayerControl().add_to(viz)
viz

### The growth order matters

An interactive visualization shows the growth order is below. It matters if a city has a limited budget. The default growth strategy is by betweenness, which is a proxy for flow. Our research has shown it is better than other strategies.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt

step = widgets.IntSlider(
    value=0, min=0, max=len(edges_ordered), step=1, description="Growth step:", layout=widgets.Layout(width='600px')
)
def update_map(step=step):
    fig, ax = plt.subplots(figsize=(8, 6))
    edges_ordered[edges_ordered["ordering"] <= step].plot(
        linewidth=3,
        color="#096a51",
        ax=ax,
    )
    edges_ordered[edges_ordered["ordering"] == 0].plot(
        linewidth=3.5,
        color="#9999cc",
        ax=ax,
    )
    ax.set_title(f"Nyborg bike net growth, growth step {step}", fontsize=12)
    ax.set_ylim(edges_ordered.total_bounds[1], edges_ordered.total_bounds[3])
    ax.set_xlim(edges_ordered.total_bounds[0], edges_ordered.total_bounds[2])
    plt.axis('off')
    plt.show()

widgets.interactive(update_map, step=step)

GrowBikeNet allows to export plots of each growth stage and to stitch them together in a video.

## Outlook

There are many more use cases and settings for GrowBikeNet. Explore them in our Usage docs: [https://docs.bikenetkit.org/GrowBikeNet/usage/](https://docs.bikenetkit.org/GrowBikeNet/usage/)

BikeNetKit also consists of more tools, like [LinkBikeNet](https://github.com/BikeNetKit/LinkBikeNet), [FixBikeNet](https://github.com/BikeNetKit/FixBikeNet), or [superblockify](https://github.com/BikeNetKit/superblockify).